In [1]:
# ---- Step 1: turn the raw QASM text into a clean list of lines ----
def clean_lines(text):
    lines = []
    for line in text.split("\n"):
        line = line.split("//")[0].strip()   # drop comments, drop whitespace
        if line:                              # skip blank lines
            lines.append(line)
    return lines


# ---- Step 2: read register sizes, e.g. "qreg q[3];" -> {"q": 3} ----
def get_registers(lines):
    registers = {}
    for line in lines:
        if line.startswith("qreg") or line.startswith("creg"):
            name = line.split()[1].split("[")[0]
            size = int(line.split("[")[1].split("]")[0])
            registers[name] = size
    return registers


# ---- Step 3: if there's a custom gate definition (like "gate syndrome ... {...}"),
#      pull out its name, its parameter names, and its body statements ----
def get_gate_definition(lines):
    for i, line in enumerate(lines):
        if line.startswith("gate "):
            header = line.split("{")[0]                  # "gate syndrome d1,d2,d3,a1,a2 "
            gate_name = header.split()[1]
            params = header.split()[2].split(",")

            body_text = ""
            for body_line in lines[i:]:
                body_text += body_line
                if "}" in body_line:
                    break
            body_text = body_text.split("{", 1)[1].split("}", 1)[0]
            body = [s.strip() for s in body_text.split(";") if s.strip()]
            return gate_name, params, body
    return None, None, None


# ---- Step 4: replace a call to the custom gate with its expanded statements,
#      substituting the call's real qubits for the definition's parameter names ----
def expand_gate_call(call_line, gate_name, params, body):
    args = call_line.split(gate_name, 1)[1].strip(" ;").split(",")
    args = [a.strip() for a in args]
    substitution = dict(zip(params, args))

    expanded = []
    for stmt in body:
        for param, real_qubit in substitution.items():
            stmt = stmt.replace(param, real_qubit)
        expanded.append(stmt)
    return expanded


# ---- Step 5: build one flat list of real statements: drop declarations,
#      inline any custom gate call, expand whole-register measures ----
def get_flat_statements(lines, registers):
    gate_name, params, body = get_gate_definition(lines)

    flat = []
    inside_gate_def = False
    for line in lines:
        if line.startswith("OPENQASM") or line.startswith("include"):
            continue
        if line.startswith("qreg") or line.startswith("creg"):
            continue
        if line.startswith("gate "):
            inside_gate_def = "}" not in line
            continue
        if inside_gate_def:
            if "}" in line:
                inside_gate_def = False
            continue

        for stmt in line.split(";"):
            stmt = stmt.strip()
            if not stmt:
                continue

            if gate_name and stmt.startswith(gate_name + " "):
                flat.extend(expand_gate_call(stmt, gate_name, params, body))
                continue

            if stmt.startswith("measure"):
                # "measure a -> syn"  ->  split into per-qubit lines if needed
                qubit_arg = stmt.split()[1]
                bit_arg = stmt.split("->")[1].strip()
                if "[" not in qubit_arg and qubit_arg in registers:
                    size = registers[qubit_arg]
                    for i in range(size):
                        flat.append(f"measure {qubit_arg}[{i}] -> {bit_arg}[{i}]")
                    continue

            flat.append(stmt)
    return flat


# ---- Step 6: find which classical registers get referenced by an "if" ----
def get_syndrome_registers(flat_statements):
    syndrome_registers = set()
    for stmt in flat_statements:
        if stmt.startswith("if("):
            reg_name = stmt.split("(")[1].split("==")[0].strip()
            syndrome_registers.add(reg_name)
    return syndrome_registers


# ---- Step 7: walk through and find stabilizer checks ----
def find_stabilizer_checks(flat_statements):
    syndrome_registers = get_syndrome_registers(flat_statements)

    pending_targets = {}   # ancilla qubit -> list of data qubits that cx'd into it
    checks = []
    check_num = 0

    for stmt in flat_statements:
        if stmt.startswith("cx"):
            qubits = stmt.replace("cx", "").strip().split(",")
            source, target = qubits[0].strip(), qubits[1].strip()
            pending_targets.setdefault(target, []).append(source)

        elif stmt.startswith("measure"):
            qubit = stmt.split()[1]
            bit = stmt.split("->")[1].strip()
            reg_name = bit.split("[")[0]

            if qubit in pending_targets and reg_name in syndrome_registers:
                checks.append({
                    "check_id": f"check_{check_num}",
                    "ancilla_qubit": qubit,
                    "data_qubits": pending_targets[qubit],
                    "syndrome_bit": bit,
                })
                check_num += 1
            pending_targets.pop(qubit, None)

    return checks


def extract_stabilizer_checks(qasm_text):
    lines = clean_lines(qasm_text)
    registers = get_registers(lines)
    flat_statements = get_flat_statements(lines, registers)
    return find_stabilizer_checks(flat_statements)

print("Parser loaded.")

Parser loaded.


In [2]:
import io, zipfile
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()
client = minio_client(settings)

key = "bronze/source=qasmbench/qasmbench-qec.zip"
response = client.get_object(settings.s3_bucket, key)
zip_bytes = response.read()
response.close()
response.release_conn()

circuits = [
    ("error_correctiond3_n5", "source", "small/error_correctiond3_n5/error_correctiond3_n5.qasm"),
    ("error_correctiond3_n5", "transpiled", "small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm"),
    ("qec_en_n5", "source", "small/qec_en_n5/qec_en_n5.qasm"),
    ("qec_en_n5", "transpiled", "small/qec_en_n5/qec_en_n5_transpiled.qasm"),
    ("qec_sm_n5", "source", "small/qec_sm_n5/qec_sm_n5.qasm"),
    ("qec_sm_n5", "transpiled", "small/qec_sm_n5/qec_sm_n5_transpiled.qasm"),
]

stabilizer_check_rows = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for benchmark, variant, member in circuits:
        circuit_id = f"{benchmark}:{variant}"
        qasm_text = z.open(member).read().decode()
        checks = extract_stabilizer_checks(qasm_text)
        print(f"{circuit_id:35s} checks={len(checks)}")

        for check in checks:
            stabilizer_check_rows.append({
                "source_record_id": f"qasmbench:{member}:{check['check_id']}",
                "circuit_id": circuit_id,
                **check,
            })

error_correctiond3_n5:source        checks=0
error_correctiond3_n5:transpiled    checks=0
qec_en_n5:source                    checks=0
qec_en_n5:transpiled                checks=0
qec_sm_n5:source                    checks=2
qec_sm_n5:transpiled                checks=2


In [3]:
import pandas as pd
stabilizer_check_df = pd.DataFrame(stabilizer_check_rows)
display(stabilizer_check_df)

,source_record_id,circuit_id,check_id,ancilla_qubit,data_qubits,syndrome_bit
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_0,qec_sm_n5:source,check_0,a[0],"[q[0], q[1]]",syn[0]
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_1,qec_sm_n5:source,check_1,a[1],"[q[1], q[2]]",syn[1]
2,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_0,a[0],"[q[0], q[1]]",syn[0]
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_1,a[1],"[q[1], q[2]]",syn[1]
